In [1]:
import pandas as pd
import os
os.chdir("../data/CHIP/")

In [7]:
ctcf_mcf7= pd.read_csv("CTCF_MCF7.bed", sep="\t", header=None)

In [9]:
ctcf_mcf7.columns = ["chrom", "start", "end", "name", "score", "strand", "signalValue", "pValue", "qValue", "peak"]

In [ ]:
ctcf_t47d = pd.read_csv("ENCODE/CTCF_T47D.bed", sep="\t")
ctcf_t47d.drop(columns=["cell_type", "tf"], inplace=True)
ctcf_t47d["tf"] = "CTCF"
ctcf_t47d["cell_type"] = "T-47D"
ctcf_t47d.to_csv("ENCODE/CTCF_T47D.bed", sep="\t", index=False)



In [47]:
import pandas as pd
import urllib.parse


def load_chip_atlas_bed(filepath):
    df = pd.read_csv(
        filepath,
        sep=r"\s+",
        header=None,
        engine="python",
        comment="t",
        on_bad_lines="skip"
    )

    if df.shape[1] != 4:
        raise ValueError(f"Expected 4 columns after parsing, got {df.shape[1]}")

    df.columns = ["chrom", "start", "end", "attributes"]

    df["experiment"] = df["attributes"].str.extract(r"ID=([^;]+)")[0]

    df["Name_clean"] = df["attributes"].str.extract(r"Name=([^;]+)")[0]
    df["Name_clean"] = df["Name_clean"].apply(
        lambda x: urllib.parse.unquote(x) if pd.notna(x) else x
    )

    df["tf"] = df["Name_clean"].str.extract(r"^\s*([^()]+?)\s*(?:\(|$)")[0].str.strip()
    df["cell_type"] = df["Name_clean"].str.extract(r"\(@\s*([^)]+)\)")[0]

    df = df.drop(columns=["Name_clean", "attributes"])

    return df

stat1 = load_chip_atlas_bed("CHIP_ATLAS/Oth.Brs.05.STAT1.AllCell.bed")
stat1.head()

,chrom,start,end,experiment,tf,cell_type
0,chr1,9997,10228,SRX1799015,STAT1,MCF-7
1,chr1,629644,629845,SRX1799015,STAT1,MCF-7
2,chr1,91387225,91387607,SRX1799015,STAT1,MCF-7
3,chr1,113812335,113812564,SRX1799015,STAT1,MCF-7
4,chr1,120176437,120176666,SRX1799015,STAT1,MCF-7


In [49]:
stat1.to_csv("CHIP_ATLAS/STAT1.bed", sep="\t", index=False)

In [51]:
stat3.to_csv("CHIP_ATLAS/STAT3.bed", sep="\t", index=False)

# UNIFY TO ONE TABLE

In [ ]:
os.chdir("\\\\wsl.localhost\\Ubuntu\\home\\stavz\\masters\\gdc\\APM\\")

from pipeline.CHIP.chip_loader import load_unified_chip
from pipeline.config import PATHS

chip = load_unified_chip(
    working_dir=PATHS.working_dir / "CHIP",
    output_path=PATHS.working_dir / "CHIP" / "unified_chip_peaks.parquet",
)

Loading ENCODE peaks from \home\stavz\masters\gdc\APM\data\CHIP\ENCODE ...
  Loaded ENCODE: CTCF_MCF7.bed (57,547 peaks)
  Loaded ENCODE: CTCF_T47D.bed (30,388 peaks)
Loading ChIP-Atlas peaks from \home\stavz\masters\gdc\APM\data\CHIP\CHIP_ATLAS ...
  Loaded CHIP_ATLAS: STAT1.bed (231 peaks)
  Loaded CHIP_ATLAS: STAT3.bed (1,308,570 peaks)


\\wsl.localhost\Ubuntu\home\stavz\masters\gdc\APM\pipeline\CHIP\chip_loader.py:166: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unified = pd.concat([encode, chip_atlas], ignore_index=True)


  Dropped 4 rows with missing coordinates

Unified ChIP table: 1,396,732 peaks
  By source:
source
CHIP_ATLAS    1308799
ENCODE          87933
  Unique TFs: 3
  Unique cell types: 16
  Saved unified table to: \home\stavz\masters\gdc\APM\data\CHIP\unified_chip_peaks.parquet
